# Debug Notebook: Compact Ablation for Qwen3 4B / 0.6B (Quick Run)

This notebook isolates dominant factors for speculative decoding performance for the Qwen3 setup by varying:
- `k` (proposal length)
- output length (`max_new_tokens`)
- prompt length bucket (short / medium / long)

Design goal: fast, GPU-only debugging run with Qwen3 4B target and Qwen3 0.6B draft, while keeping outputs compact but informative.

In [1]:
import gc
import os
import sys
from pathlib import Path

import pandas as pd
try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "torch is required for this notebook. Select a kernel/environment with PyTorch installed."
    ) from exc

# Resolve project root robustly.
search_roots = []
if '__file__' in globals():
    search_roots.append(Path(__file__).resolve().parent)
search_roots.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_roots:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/.')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior.
os.environ['SPECDEC_HF_OFFLINE_FIRST'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# GPU-only enforcement.
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError('CUDA GPU is required for this notebook.')

from config import TARGET_MODEL_ID, DRAFT_MODELS, REGIMES
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample

results_dir = project_root / 'Review' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print('GPU-only mode enabled')
print(f'CUDA device count: {torch.cuda.device_count()}')

Project root: C:\Working\speculative-decoding-main_v10\speculative-decoding-main
GPU-only mode enabled
CUDA device count: 2


In [2]:
import importlib
import os
import sys

review_root = project_root / "Review"
review_results_dir = review_root / "results"
review_figures_dir = review_root / "figures"
review_stability_dir = review_results_dir / "stability"
review_drifter_dir = review_results_dir / "drifter_ckpt"

qwen3_target_model_id = "Qwen/Qwen3-4B"
qwen3_draft_model_id = "Qwen/Qwen3-0.6B"

os.environ["SPECDEC_TARGET_MODEL_ID"] = qwen3_target_model_id
os.environ["SPECDEC_DRAFT_MODEL_ID"] = qwen3_draft_model_id
os.environ["SPECDEC_RESULTS_DIR"] = str(review_results_dir)
os.environ["SPECDEC_STABILITY_DIR"] = str(review_stability_dir)
os.environ["SPECDEC_FIGURES_DIR"] = str(review_figures_dir)
os.environ["SPECDEC_DRIFTER_CHECKPOINT_DIR"] = str(review_drifter_dir)
os.environ["SPECDEC_TARGET_QUANT"] = "fp16"
os.environ["SPECDEC_DRAFT_QUANT"] = "fp16"
os.environ["SPECDEC_ACSD_PRIMARY_DRAFT"] = "0.6B"
os.environ["SPECDEC_ACSD_RESCUE_DRAFT"] = "0.6B"

config_module = sys.modules.get("config")
if config_module is None:
    import config as config_module
config_module = importlib.reload(config_module)

globals().update({
    "TARGET_MODEL_ID": config_module.TARGET_MODEL_ID,
    "DRAFT_MODELS": config_module.DRAFT_MODELS,
    "RESULTS_DIR": config_module.RESULTS_DIR,
    "STABILITY_DIR": config_module.STABILITY_DIR,
    "FIGURES_DIR": config_module.FIGURES_DIR,
    "TARGET_QUANT": config_module.TARGET_QUANT,
    "DRAFT_QUANT": config_module.DRAFT_QUANT,
})
if hasattr(config_module, "ACSD_EVAL"):
    globals()["ACSD_EVAL"] = config_module.ACSD_EVAL

for module_name in ("runtime", "baseline", "speculative", "acsd", "hf_assistant", "drift_speculative"):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

In [3]:
# Ablation controls (compact Qwen3 quick run).
DEVICE = 'cuda:0'
TARGET_QUANT = 'fp16'
DRAFT_QUANT = 'fp16'
DRAFT_LABEL = '0.6B'
MODES = ['Deterministic']  # add 'Stochastic' if needed

K_VALUES = [2, 4, 8]
OUTPUT_LENGTHS = [12, 32, 64]
PROMPT_BUCKETS = ['short', 'medium', 'long']

BASE_PROMPTS = [
    'Why are leaves green? Answer in one sentence.',
    'What is an API? One-sentence answer.',
    'Define overfitting in machine learning in one sentence.',
    'Give one concise writing tip.',
]

N_PROMPTS_PER_BUCKET = 2

RAW_CSV = results_dir / 'ablation_compact_raw.csv'
SUMMARY_CSV = results_dir / 'ablation_compact_summary.csv'
DOMINANCE_CSV = results_dir / 'ablation_factor_dominance.csv'

print('Ablation settings:')
print(f'  device={DEVICE}, target_quant={TARGET_QUANT}, draft_quant={DRAFT_QUANT}, draft={DRAFT_LABEL}')
print(f'  target={TARGET_MODEL_ID}')
print(f'  draft_model={DRAFT_MODELS[DRAFT_LABEL]}')
print(f'  modes={MODES}, k={K_VALUES}, output_lengths={OUTPUT_LENGTHS}, buckets={PROMPT_BUCKETS}')

Ablation settings:
  device=cuda:0, target_quant=fp16, draft_quant=fp16, draft=0.6B
  target=Qwen/Qwen3-4B
  draft_model=Qwen/Qwen3-0.6B
  modes=['Deterministic'], k=[2, 4, 8], output_lengths=[12, 32, 64], buckets=['short', 'medium', 'long']


In [4]:
# Prompt-bucket constructor (vary prompt length while preserving task intent).
MEDIUM_PREFIX = (
    'Context: In practical ML systems, response quality, latency, and resource usage must be balanced. '
    'Provide a concise, technically accurate answer. '
)
LONG_PREFIX = ' '.join([
    'Context: System constraints include memory bandwidth, kernel launch overhead, and cache behavior.'
    'Prompt engineering can alter prefill cost and token distribution.'
    'Focus on clarity and avoid unnecessary verbosity.'
] * 8)

def make_prompt(base: str, bucket: str) -> str:
    if bucket == 'short':
        return base
    if bucket == 'medium':
        return f'{MEDIUM_PREFIX}\nQuestion: {base}'
    if bucket == 'long':
        return f'{LONG_PREFIX}\nQuestion: {base}'
    raise ValueError(f'Unknown bucket: {bucket}')

bucket_prompts = {}
for b in PROMPT_BUCKETS:
    bucket_prompts[b] = [make_prompt(p, b) for p in BASE_PROMPTS[:N_PROMPTS_PER_BUCKET]]

for b, ps in bucket_prompts.items():
    print(f'{b}: {len(ps)} prompts')

short: 2 prompts
medium: 2 prompts
long: 2 prompts


In [5]:
# Model caches and single-run helper.
target_cache = {}
draft_cache = {}

def _get_target(quant: str):
    key = (TARGET_MODEL_ID, quant.lower(), DEVICE)
    if key not in target_cache:
        target_cache[key] = load_model_on_device(TARGET_MODEL_ID, device=DEVICE, quant_mode=quant.lower())
    return target_cache[key]

def _get_draft(quant: str):
    model_id = DRAFT_MODELS[DRAFT_LABEL]
    key = (model_id, quant.lower(), DEVICE)
    if key not in draft_cache:
        draft_cache[key] = load_model_on_device(model_id, device=DEVICE, quant_mode=quant.lower())
    return draft_cache[key]

def run_one(prompt: str, mode: str, k: int, max_new_tokens: int) -> dict:
    regime = REGIMES[mode.lower()]

    target_model, target_tok = _get_target(TARGET_QUANT)
    draft_model, _ = _get_draft(DRAFT_QUANT)

    # Prompt token length (prefill proxy).
    prompt_tokens = int(target_tok(prompt, return_tensors='pt', truncation=True, max_length=2048)['input_ids'].shape[1])

    base = run_baseline_sample(
        target_model,
        target_tok,
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=regime.temperature,
        top_p=regime.top_p,
    )

    spec = speculative_decode_sample(
        target_model,
        draft_model,
        target_tok,
        prompt,
        max_new_tokens=max_new_tokens,
        k=k,
        temperature=regime.temperature,
        top_p=regime.top_p,
        return_timing_breakdown=True,
    )

    base_s = float(base.get('latency_s', 0.0))
    spec_s = float(spec.get('latency_s', 0.0))
    draft_s = float(spec.get('draft_elapsed_s', 0.0))
    verify_s = float(spec.get('verify_elapsed_s', 0.0))
    other_s = max(spec_s - draft_s - verify_s, 0.0)

    proposed = float(spec.get('total_proposed', 0.0))
    accepted = float(spec.get('total_accepted', 0.0))
    alpha = (accepted / proposed) if proposed > 0 else 0.0
    speedup = (base_s / spec_s) if spec_s > 0 else 0.0

    return {
        'prompt_tokens': prompt_tokens,
        'baseline_latency_s': base_s,
        'spec_latency_s': spec_s,
        'speedup': speedup,
        'alpha': alpha,
        'draft_s': draft_s,
        'verify_s': verify_s,
        'other_s': other_s,
        'draft_share': (draft_s / spec_s) if spec_s > 0 else 0.0,
        'verify_share': (verify_s / spec_s) if spec_s > 0 else 0.0,
        'other_share': (other_s / spec_s) if spec_s > 0 else 0.0,
    }

In [6]:
# Execute compact ablation grid.
rows = []
total_runs = len(MODES) * len(K_VALUES) * len(OUTPUT_LENGTHS) * len(PROMPT_BUCKETS) * N_PROMPTS_PER_BUCKET
done = 0

for mode in MODES:
    for k in K_VALUES:
        for out_len in OUTPUT_LENGTHS:
            for bucket in PROMPT_BUCKETS:
                prompts = bucket_prompts[bucket]
                for i, prompt in enumerate(prompts, start=1):
                    done += 1
                    print(f'[{done}/{total_runs}] mode={mode} k={k} out={out_len} bucket={bucket} sample={i}')
                    m = run_one(prompt=prompt, mode=mode, k=k, max_new_tokens=out_len)
                    rows.append({
                        'mode': mode,
                        'k': k,
                        'max_new_tokens': out_len,
                        'prompt_bucket': bucket,
                        'sample_idx': i,
                        **m,
                    })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(RAW_CSV, index=False)
print(f'Saved raw runs -> {RAW_CSV}')
display(df_raw.head())

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

[1/54] mode=Deterministic k=2 out=12 bucket=short sample=1
Loading model: Qwen/Qwen3-4B (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading model: Qwen/Qwen3-0.6B (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[2/54] mode=Deterministic k=2 out=12 bucket=short sample=2
[3/54] mode=Deterministic k=2 out=12 bucket=medium sample=1
[4/54] mode=Deterministic k=2 out=12 bucket=medium sample=2
[5/54] mode=Deterministic k=2 out=12 bucket=long sample=1
[6/54] mode=Deterministic k=2 out=12 bucket=long sample=2
[7/54] mode=Deterministic k=2 out=32 bucket=short sample=1
[8/54] mode=Deterministic k=2 out=32 bucket=short sample=2
[9/54] mode=Deterministic k=2 out=32 bucket=medium sample=1
[10/54] mode=Deterministic k=2 out=32 bucket=medium sample=2
[11/54] mode=Deterministic k=2 out=32 bucket=long sample=1
[12/54] mode=Deterministic k=2 out=32 bucket=long sample=2
[13/54] mode=Deterministic k=2 out=64 bucket=short sample=1
[14/54] mode=Deterministic k=2 out=64 bucket=short sample=2
[15/54] mode=Deterministic k=2 out=64 bucket=medium sample=1
[16/54] mode=Deterministic k=2 out=64 bucket=medium sample=2
[17/54] mode=Deterministic k=2 out=64 bucket=long sample=1
[18/54] mode=Deterministic k=2 out=64 bucket=lo

,mode,k,max_new_tokens,prompt_bucket,sample_idx,prompt_tokens,baseline_latency_s,spec_latency_s,speedup,alpha,draft_s,verify_s,other_s,draft_share,verify_share,other_share
0,Deterministic,2,12,short,1,10,0.5662,0.5554,1.019445,0.333333,0.291535,0.233028,0.030837,0.524910,0.419568,0.055522
1,Deterministic,2,12,short,2,10,0.2954,0.3798,0.777778,0.800000,0.190217,0.183721,0.005862,0.500835,0.483731,0.015434
2,Deterministic,2,12,medium,1,40,0.3008,0.4925,0.610761,0.357143,0.262127,0.221632,0.008741,0.532238,0.450014,0.017748
3,Deterministic,2,12,medium,2,40,0.2962,0.5289,0.560030,0.266667,0.282011,0.236032,0.010857,0.533203,0.446270,0.020528
4,Deterministic,2,12,long,1,284,0.3053,0.4083,0.747735,0.700000,0.188800,0.213212,0.006288,0.462405,0.522194,0.015400


In [7]:
# Compact ablation summary table.
group_cols = ['mode', 'k', 'max_new_tokens', 'prompt_bucket']
summary = (
    df_raw.groupby(group_cols, as_index=False)
    .agg(
        n=('sample_idx', 'count'),
        prompt_tokens_mean=('prompt_tokens', 'mean'),
        speedup_mean=('speedup', 'mean'),
        alpha_mean=('alpha', 'mean'),
        spec_latency_mean_s=('spec_latency_s', 'mean'),
        baseline_latency_mean_s=('baseline_latency_s', 'mean'),
        draft_share_mean=('draft_share', 'mean'),
        verify_share_mean=('verify_share', 'mean'),
        other_share_mean=('other_share', 'mean'),
    )
)

for c in ['draft_share_mean', 'verify_share_mean', 'other_share_mean']:
    summary[c] = summary[c] * 100.0

summary.to_csv(SUMMARY_CSV, index=False)
print(f'Saved summary -> {SUMMARY_CSV}')
display(summary.sort_values(['mode', 'k', 'max_new_tokens', 'prompt_bucket']))

Saved summary -> C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\ablation_compact_summary.csv


,mode,k,max_new_tokens,prompt_bucket,n,prompt_tokens_mean,speedup_mean,alpha_mean,spec_latency_mean_s,baseline_latency_mean_s,draft_share_mean,verify_share_mean,other_share_mean
0,Deterministic,2,12,long,2,284.0,0.669615,0.528571,0.46400,0.30635,48.458815,49.993695,1.547490
1,Deterministic,2,12,medium,2,40.0,0.585396,0.311905,0.51070,0.29850,53.272022,44.814191,1.913787
2,Deterministic,2,12,short,2,10.0,0.898612,0.566667,0.46760,0.43080,51.287231,45.164940,3.547829
3,Deterministic,2,32,long,2,284.0,0.755596,0.469697,1.03640,0.78305,60.167540,37.845019,1.987441
4,Deterministic,2,32,medium,2,40.0,0.644280,0.262053,1.23240,0.79375,63.924114,34.006211,2.069675
5,Deterministic,2,32,short,2,10.0,0.888257,0.622619,0.88545,0.78545,62.207821,35.884664,1.907515
6,Deterministic,2,64,long,2,284.0,0.723121,0.395076,2.17380,1.55980,66.810607,31.206245,1.983148
7,Deterministic,2,64,medium,2,40.0,0.739648,0.333590,2.12345,1.56960,67.299565,30.637093,2.063342
8,Deterministic,2,64,short,2,10.0,0.912375,0.554238,1.72535,1.56365,66.401631,31.555414,2.042955
9,Deterministic,4,12,long,2,284.0,0.621952,0.581448,0.49765,0.30640,56.352272,42.437522,1.210207


In [8]:
# Dominance scores: how much variance each factor explains (main effects only).
def dominance_score(df: pd.DataFrame, y: str, factor: str) -> float:
    total_var = float(df[y].var(ddof=0))
    if total_var <= 0:
        return 0.0
    between = float(df.groupby(factor)[y].mean().var(ddof=0))
    return between / total_var

factors = ['k', 'max_new_tokens', 'prompt_bucket']
targets = ['speedup', 'spec_latency_s', 'alpha']
dom_rows = []
for y in targets:
    for f in factors:
        dom_rows.append({
            'metric': y,
            'factor': f,
            'dominance_score': dominance_score(df_raw, y, f),
        })

df_dom = pd.DataFrame(dom_rows)
df_dom.to_csv(DOMINANCE_CSV, index=False)
print(f'Saved dominance -> {DOMINANCE_CSV}')
display(df_dom.sort_values(['metric', 'dominance_score'], ascending=[True, False]))

print('\nHigher dominance_score means stronger main-effect contribution to variance.')

Saved dominance -> C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\ablation_factor_dominance.csv


,metric,factor,dominance_score
8,alpha,prompt_bucket,0.326489
7,alpha,max_new_tokens,0.036695
6,alpha,k,0.017605
4,spec_latency_s,max_new_tokens,0.714927
3,spec_latency_s,k,0.053091
5,spec_latency_s,prompt_bucket,0.043577
0,speedup,k,0.277163
2,speedup,prompt_bucket,0.206110
1,speedup,max_new_tokens,0.019945



Higher dominance_score means stronger main-effect contribution to variance.


## Notes

- This notebook is for compact diagnosis, not final benchmark claims.
- To run faster: reduce `N_PROMPTS_PER_BUCKET` or fewer `OUTPUT_LENGTHS`.
- To compare regimes, set `MODES = ['Deterministic', 'Stochastic']`.
- You can reuse outputs in slides directly from:
  - `results/ablation_compact_summary.csv`
  - `results/ablation_factor_dominance.csv`